<a href="https://colab.research.google.com/github/yourmom2626/Rescuenet/blob/main/Copy_of_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

plantseg = load_dataset(
    "LFTTim/plantseg_disease_segmentation",
    streaming=True
)

print(plantseg)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample["image"])
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sample["mask"])
plt.title("Disease Segmentation Mask")
plt.axis("off")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

tomato_samples = []

for item in plantseg["train"]:
    if "tomato" in item["plant"].lower():
        tomato_samples.append(item)

    if len(tomato_samples) == 3:
        break

print("Tomato samples found:", len(tomato_samples))

for i, item in enumerate(tomato_samples):
    print("Plant:", item["plant"])
    print("Disease:", item["disease"])

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(item["image"])
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(item["mask"])
    plt.title("Disease Mask")
    plt.axis("off")

    plt.show()

In [ ]:
!git clone https://github.com/yourmom2626/Rescuenet.git
%cd Rescuenet
!ls -la

In [ ]:
import os
# Check if Rescuenet directory exists, if not clone it
if not os.path.exists('Rescuenet'):
    !git clone https://github.com/yourmom2626/Rescuenet.git

%cd Rescuenet
!ls -la

In [ ]:
import os
import numpy as np
from PIL import Image
from datasets import load_dataset

# Create target directories
base_dir = "corn_dataset"
splits = ["train", "validation", "test"]

for s in splits:
    os.makedirs(os.path.join(base_dir, s, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, s, "masks"), exist_ok=True)

print("Downloading and filtering Hugging Face dataset for corn...")
# Load dataset non-streaming to make split reading robust and complete
hf_dataset = load_dataset("LFTTim/plantseg_disease_segmentation")

counts = {"train": 0, "validation": 0, "test": 0}

# Map dataset splits to target directories
# Note: Hugging Face dataset splits vs expected destination
for split_name in hf_dataset.keys():
    for idx, item in enumerate(hf_dataset[split_name]):
        plant_type = str(item.get("plant", "")).lower()
        # We only want corn images
        if "corn" in plant_type or "maize" in plant_type:
            # Determine actual project split alignment
            item_split = item.get("split", "train").lower()
            if item_split == "val":
                dest_split = "validation"
            elif item_split == "test":
                dest_split = "test"
            else:
                dest_split = "train"

            img = item["image"]
            mask = item["mask"]

            # Convert image to RGB if not already
            if img.mode != "RGB":
                img = img.convert("RGB")

            # Convert mask to L (grayscale) and then binary (0 and 255)
            mask_np = np.array(mask)
            binary_mask = np.where(mask_np > 0, 255, 0).astype(np.uint8)
            mask_img = Image.fromarray(binary_mask)

            img_filename = f"corn_{idx}.png"
            mask_filename = f"corn_{idx}.png"

            img.save(os.path.join(base_dir, dest_split, "images", img_filename))
            mask_img.save(os.path.join(base_dir, dest_split, "masks", mask_filename))
            counts[dest_split] += 1

print("Extraction completed!")
print("Extracted file counts:", counts)

README.md:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  393MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  471MB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  405MB            

data/train-00002-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  449MB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11458 [00:00<?, ? examples/s]